# 🗄️ updateDB — Cập nhật & Xoá Dữ liệu ChromaDB

Notebook kiểm thử **trước khi chuyển thành module** chức năng:

| Chức năng | Mô tả |
|-----------|-------|
| **Ingest mới** | Nhận văn bản OCR → tiền xử lý → tách Điều → chunking → upsert vào ChromaDB |
| **Auto-detect bãi bỏ** | Quét văn bản mới xem có bãi bỏ/hết hiệu lực văn bản nào → xoá khỏi ChromaDB |
| **Xoá thủ công** | Nhập số hiệu văn bản hoặc điều khoản cụ thể để xoá |
| **Verify** | Kiểm tra trước/sau mỗi thao tác |

> ⚠️ **Notebook này không modify BM25 index** — BM25 sẽ được rebuild thủ công khi cần thiết (xem Cell cuối).

---

## 0. Imports & Config

In [ ]:
import re
import json
import time
import hashlib
import unicodedata
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
print("✅ Imports OK")
print(f"   Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# ============================================================
# CONFIG — chỉnh sửa ở đây nếu cần
# ============================================================
PROJECT_ROOT = Path(".").resolve()   # thay bằng đường dẫn project nếu cần
CHUNK_DIR    = PROJECT_ROOT / "dataset" / "chunks"
CHROMA_DIR   = PROJECT_ROOT / "dataset" / "chromadb"
MODEL_PATH   = PROJECT_ROOT / "models" / "embedding"

EMBED_MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
EMBED_DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_BATCH_SIZE = 32  # thấp hơn embedAndIndex vì chạy local

COLLECTION_NAMES = {
    "legal"   : "legal_chunks",
    "forms"   : "forms_chunks",
    "examples": "examples_chunks",
}

# Chunking params — giống chunkData.ipynb
LEGAL_MAX_WORDS     = 400
LEGAL_MIN_WORDS     = 30
LEGAL_OVERLAP_WORDS = 50

# Sanity check paths
for name, path in {"chromadb": CHROMA_DIR, "chunks": CHUNK_DIR}.items():
    status = "✅" if path.exists() else "❌ KHÔNG TÌM THẤY"
    print(f"  [{status}] {name}: {path}")

## 1. Khởi tạo ChromaDB & Embedding Model

In [ ]:
# ── ChromaDB client ──────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

def get_collection(name: str) -> chromadb.Collection:
    return chroma_client.get_collection(name=name)

# Kiểm tra collections hiện có
print("📦 ChromaDB collections:")
for key, col_name in COLLECTION_NAMES.items():
    try:
        col = get_collection(col_name)
        print(f"  ✅ {col_name}: {col.count():,} chunks")
    except Exception as e:
        print(f"  ❌ {col_name}: {e}")

In [ ]:
# ── Embedding model ──────────────────────────────────────────────────────────
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
print("(Có thể mất 1-2 phút lần đầu nếu chưa cache)")
t0 = time.time()

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=EMBED_DEVICE,
    cache_folder=str(MODEL_PATH),
)

print(f"✅ Model loaded in {time.time()-t0:.1f}s")

def embed_passages(texts: List[str]) -> np.ndarray:
    """Embed passages để index (prefix 'passage: ')."""
    prefixed = [f"passage: {t}" for t in texts]
    return embed_model.encode(
        prefixed, batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=len(texts) > 10,
    )

# Smoke test
_v = embed_passages(["Điều 1. Phạm vi điều chỉnh."])
print(f"   Embed shape: {_v.shape}  ✅")

---
## 2. Utility Functions — Dùng chung cho toàn notebook

In [ ]:
# ============================================================
# 2.1 ChromaDB ID helpers (giống embedAndIndex.ipynb)
# ============================================================

def make_chroma_id(doc_id: str, chunk_index: int, text: str) -> str:
    """Tái tạo ChromaDB ID giống hệt embedAndIndex.ipynb."""
    raw = f"{doc_id}|{chunk_index}|{text[:200]}"
    return hashlib.sha256(raw.encode()).hexdigest()[:24]


def coerce_metadata(meta: Dict) -> Dict:
    """Chuyển metadata về kiểu ChromaDB-compatible."""
    clean: Dict = {}
    for k, v in meta.items():
        if v is None:
            continue
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        elif isinstance(v, (np.bool_,)):
            clean[k] = bool(v)
        elif isinstance(v, (list, dict)):
            clean[k] = json.dumps(v, ensure_ascii=False)
        elif isinstance(v, (str, int, float, bool)):
            clean[k] = v
        else:
            clean[k] = str(v)
    return clean


# ============================================================
# 2.2 Text utils
# ============================================================

def count_words(text: str) -> int:
    return len(text.split()) if isinstance(text, str) else 0


def normalize_text(text: str) -> str:
    """Chuẩn hoá whitespace, newline, tab — giống preprocessData."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def make_doc_id(so_hieu: str, article: str, content: str) -> str:
    """Tạo doc_id = SHA256(số_hiệu + article + content_hash[:10]) — giống preprocessData."""
    content_hash = hashlib.sha256(content.encode()).hexdigest()[:10]
    raw = f"{so_hieu}|{article}|{content_hash}"
    return hashlib.sha256(raw.encode()).hexdigest()[:24]


print("✅ Utility functions defined.")

---
## 3. Tiền xử lý & Phân tách Điều từ văn bản OCR

In [ ]:
# ============================================================
# 3.1 Tách cấu trúc văn bản OCR → danh sách Điều
# ============================================================

# Pattern nhận diện đầu Điều trong luật VN
ARTICLE_PATTERN = re.compile(
    r'(?:^|\n)\s*'           # đầu dòng
    r'(Điều\s+\d+[a-z]?\.?'  # "Điều 1." hoặc "Điều 1a."
    r'(?:\s+[^\n]+)?)',        # tiêu đề điều (tùy chọn)
    re.MULTILINE | re.UNICODE
)

# Pattern nhận diện đầu Chương
CHAPTER_PATTERN = re.compile(
    r'(?:^|\n)\s*(Chương\s+[IVXLCDM]+\.?(?:\s+[^\n]+)?)',
    re.MULTILINE | re.UNICODE
)


def extract_doc_header(text: str) -> Dict[str, str]:
    """
    Trích metadata từ phần header văn bản OCR.
    Trả về dict: {so_hieu, ten_van_ban, co_quan, loai_vb}
    """
    header = {"so_hieu": "", "ten_van_ban": "", "co_quan": "", "loai_vb": "KHÔNG RÕ"}
    lines = text[:2000].split("\n")

    # Số hiệu: "123/2024/NĐ-CP", "15/2020/QH14", v.v.
    so_hieu_pat = re.compile(
        r'\b(\d{1,3}/\d{4}/[A-ZĐÔƯĂ][A-ZĐÔƯĂ\-]+(?:/[A-ZĐÔƯĂ]+)?)',
        re.UNICODE
    )
    for line in lines:
        m = so_hieu_pat.search(line)
        if m:
            header["so_hieu"] = m.group(1)
            break

    # Loại văn bản
    loai_map = [
        (r'LUẬT', 'LUẬT'),
        (r'NGHỊ ĐỊNH', 'NGHỊ ĐỊNH'),
        (r'NGHỊ QUYẾT', 'NGHỊ QUYẾT'),
        (r'THÔNG TƯ', 'THÔNG TƯ'),
        (r'QUYẾT ĐỊNH', 'QUYẾT ĐỊNH'),
        (r'PHÁP LỆNH', 'PHÁP LỆNH'),
    ]
    for pat, loai in loai_map:
        if re.search(pat, text[:500], re.IGNORECASE):
            header["loai_vb"] = loai
            break

    # Tên văn bản: thường là dòng IN HOA hoặc dòng sau "VỀ"
    for i, line in enumerate(lines[:30]):
        line_clean = line.strip()
        if len(line_clean) > 20 and re.search(r'[VỀ|QUY ĐỊNH|HƯỚNG DẪN|BAN HÀNH]', line_clean, re.IGNORECASE):
            header["ten_van_ban"] = line_clean[:200]
            break

    return header


def split_articles(text: str) -> List[Dict[str, str]]:
    """
    Tách văn bản OCR thành danh sách các Điều.
    Mỗi phần tử: {article, content, chapter_id, chapter_name}
    """
    parts = ARTICLE_PATTERN.split(text)

    if len(parts) <= 1:
        # Không tìm thấy Điều nào — coi cả văn bản là 1 điều
        return [{"article": "Toàn văn", "content": text.strip(),
                 "chapter_id": "", "chapter_name": ""}]

    articles = []
    # Theo dõi chương hiện tại
    current_chapter_id   = ""
    current_chapter_name = ""

    # parts = [pre_text, article_header1, content1, article_header2, content2, ...]
    i = 1
    while i < len(parts) - 1:
        article_header = parts[i].strip()
        raw_content    = parts[i + 1]

        # Kiểm tra xem trước content có Chương mới không
        chapter_m = CHAPTER_PATTERN.search(raw_content)
        if chapter_m:
            chap_text = chapter_m.group(1).strip()
            chap_num  = re.search(r'[IVXLCDM]+', chap_text)
            current_chapter_id   = chap_num.group(0) if chap_num else ""
            current_chapter_name = chap_text
            # Xoá dòng chương khỏi content
            raw_content = raw_content[:chapter_m.start()] + raw_content[chapter_m.end():]

        content = normalize_text(raw_content)

        if content:  # bỏ điều trống
            articles.append({
                "article"     : article_header,
                "content"     : content,
                "chapter_id"  : current_chapter_id,
                "chapter_name": current_chapter_name,
            })
        i += 2

    return articles


# ── Demo / test ───────────────────────────────────────────────────────────────
SAMPLE_OCR = """
NGHỊ ĐỊNH
Số: 30/2020/NĐ-CP
VỀ CÔNG TÁC VĂN THƯ

Chương I. QUY ĐỊNH CHUNG

Điều 1. Phạm vi điều chỉnh
Nghị định này quy định về công tác văn thư; quản lý văn bản;
lập hồ sơ và giao nộp tài liệu vào Lưu trữ cơ quan.

Điều 2. Đối tượng áp dụng
Nghị định này áp dụng đối với:
a) Cơ quan, tổ chức nhà nước;
b) Doanh nghiệp nhà nước;
c) Tổ chức chính trị, chính trị - xã hội.

Chương II. THỂ THỨC VÀ KỸ THUẬT TRÌNH BÀY VĂN BẢN

Điều 8. Các thành phần thể thức văn bản
1. Thành phần bắt buộc gồm:
a) Quốc hiệu và Tiêu ngữ;
b) Tên cơ quan, tổ chức ban hành văn bản;
c) Số, ký hiệu của văn bản.
2. Thành phần bổ sung gồm phụ lục, dấu chỉ mức độ khẩn.
"""

header = extract_doc_header(SAMPLE_OCR)
print("📋 Header văn bản:", header)

articles = split_articles(SAMPLE_OCR)
print(f"\n🔖 Tìm thấy {len(articles)} Điều:")
for a in articles:
    print(f"  [{a['chapter_id'] or '?'}] {a['article'][:60]}")
    print(f"       → {count_words(a['content'])} từ")

## 4. Chunking — Article-aware + Clause-aware (giống chunkData v2)

In [ ]:
# ============================================================
# 4.1 Clause-aware splitting (giống chunkData v2)
# ============================================================

CLAUSE_PATTERN = re.compile(
    r'(?:^|\n)'
    r'(\d{1,2}\. '
    r'|[a-zđ]\) '
    r'|[àáảãạăắặằẳẵâấậầẩẫ]\) '
    r'|- (?=\S)'
    r')',
    re.MULTILINE | re.UNICODE
)


def split_into_clauses(text: str) -> List[Tuple[str, str]]:
    parts = CLAUSE_PATTERN.split(text)
    if len(parts) <= 1:
        return [(text.strip(), "no_clause")]
    result = []
    pre = parts[0].strip()
    if pre:
        result.append((pre, "preamble"))
    i = 1
    while i < len(parts) - 1:
        label   = parts[i].strip()
        content = parts[i + 1]
        clause_text = (label + " " + content).strip()
        if clause_text:
            ctype = "numbered" if re.match(r'\d', label) else ("bullet" if label.startswith("-") else "lettered")
            result.append((clause_text, ctype))
        i += 2
    return result if result else [(text.strip(), "no_clause")]


def fixed_word_split(text: str, max_words: int, overlap_words: int) -> List[str]:
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap_words
    return chunks


def chunk_article(article_dict: Dict, so_hieu: str, loai_vb: str,
                  ten_vb: str, ministry: str = "") -> List[Dict]:
    """
    Chunking 1 Điều → list chunk dicts.
    Đầu ra tương thích với schema legal_chunks trong ChromaDB.
    """
    article = article_dict["article"]
    content = article_dict["content"]
    chapter_id   = article_dict.get("chapter_id", "")
    chapter_name = article_dict.get("chapter_name", "")

    doc_id = make_doc_id(so_hieu, article, content)

    meta_base = {
        "doc_id"         : doc_id,
        "source_doc_no"  : so_hieu,
        "ministry"       : ministry,
        "type_normalized": loai_vb,
        "doc_name"       : ten_vb,
        "chapter_id"     : chapter_id,
        "chapter_name"   : chapter_name,
        "article"        : article,
    }

    # ── CASE 1: Điều ngắn → 1 chunk ──────────────────────────────────────
    if count_words(content) <= LEGAL_MAX_WORDS:
        text = f"{article}\n{content}" if article else content
        return [{
            **meta_base,
            "chunk_index" : 0,
            "total_chunks": 1,
            "split_type"  : "article",
            "text"        : text,
            "word_count"  : count_words(text),
        }]

    # ── CASE 2: Điều dài → split theo khoản ─────────────────────────────
    clauses = split_into_clauses(content)
    raw_chunks, split_type = [], "clause"

    for clause_text, ctype in clauses:
        if count_words(clause_text) <= LEGAL_MAX_WORDS:
            raw_chunks.append(clause_text)
        else:
            sub = fixed_word_split(clause_text, LEGAL_MAX_WORDS, LEGAL_OVERLAP_WORDS)
            raw_chunks.extend(sub)
            split_type = "fixed_word"

    # Merge chunks quá ngắn
    merged = []
    for chunk in raw_chunks:
        if merged and count_words(chunk) < LEGAL_MIN_WORDS:
            merged[-1] = merged[-1] + " " + chunk
        else:
            merged.append(chunk)

    results = []
    for idx, chunk_text in enumerate(merged):
        header_prefix = f"[{article}]\n" if len(merged) > 1 else ""
        text = header_prefix + chunk_text
        results.append({
            **meta_base,
            "chunk_index" : idx,
            "total_chunks": len(merged),
            "split_type"  : split_type,
            "text"        : text,
            "word_count"  : count_words(text),
        })

    return results


print("✅ Chunking functions defined.")

## 5. Auto-detect Bãi bỏ / Hết hiệu lực

In [ ]:
# ============================================================
# 5.1 Patterns phát hiện bãi bỏ — giống obsoleteFilter.ipynb
# ============================================================

OBSOLETE_PATTERNS = [
    # Bãi bỏ kèm số hiệu
    re.compile(r'bãi bỏ\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    # Thay thế kèm số hiệu
    re.compile(r'thay thế\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    # Hết hiệu lực kèm số hiệu
    re.compile(r'hết hiệu lực\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    # Hủy bỏ / đình chỉ
    re.compile(r'(?:hủy bỏ|đình chỉ thi hành)\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    # Không còn hiệu lực
    re.compile(r'không còn hiệu lực\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
]

# Số hiệu văn bản quan trọng — KHÔNG BAO GIỜ xoá
NEVER_REMOVE = {
    "30/2020/NĐ-CP", "80/2015/QH13", "15/2020/QH14", "45/2019/QH14",
    "22/2008/QH12", "58/2010/QH12", "01/2011/QH13", "76/2015/QH13",
    "77/2015/QH13", "36/2018/QH14", "02/2011/QH13", "34/2016/NĐ-CP",
    "138/2020/NĐ-CP", "115/2020/NĐ-CP", "61/2018/NĐ-CP", "34/2019/NĐ-CP",
}


def normalize_doc_no(s: str) -> str:
    """Chuẩn hoá số hiệu để so sánh."""
    return s.strip().upper().replace(" ", "")


def detect_abolished_docs(full_text: str) -> List[Dict]:
    """
    Quét toàn bộ văn bản OCR để tìm số hiệu văn bản bị bãi bỏ.
    Trả về list: [{so_hieu, pattern_type, snippet}]
    Loại trừ các văn bản trong NEVER_REMOVE.
    """
    found: Dict[str, Dict] = {}  # so_hieu → {count, snippets, types}

    for pat in OBSOLETE_PATTERNS:
        for m in pat.finditer(full_text):
            so_hieu = normalize_doc_no(m.group(1))
            if not so_hieu or len(so_hieu) < 5:
                continue  # quá ngắn → false positive
            if so_hieu in {normalize_doc_no(x) for x in NEVER_REMOVE}:
                continue  # protected

            snippet = full_text[max(0, m.start()-30):m.end()+60].replace("\n", " ")
            if so_hieu not in found:
                found[so_hieu] = {"count": 0, "snippets": [], "pattern": pat.pattern[:50]}
            found[so_hieu]["count"] += 1
            found[so_hieu]["snippets"].append(snippet)

    # Chỉ trả về nếu xuất hiện ≥ 1 lần (giảm ngưỡng từ 2→1 cho văn bản cụ thể)
    results = []
    for so_hieu, info in found.items():
        if info["count"] >= 1:
            results.append({
                "so_hieu" : so_hieu,
                "count"   : info["count"],
                "snippet" : info["snippets"][0][:150],
                "pattern" : info["pattern"],
            })

    return results


# ── Test ──────────────────────────────────────────────────────────────────────
TEST_OBSOLETE = """
Điều 10. Hiệu lực thi hành
Nghị định này có hiệu lực từ ngày 01/03/2022.
Bãi bỏ Nghị định số 110/2004/NĐ-CP ngày 08 tháng 4 năm 2004 của Chính phủ
về công tác văn thư.
Thay thế Quyết định số 09/2010/QĐ-TTg về điều phối ngân sách.
Điều khoản về 30/2020/NĐ-CP vẫn còn hiệu lực.
"""
abolished = detect_abolished_docs(TEST_OBSOLETE)
print(f"Tìm thấy {len(abolished)} văn bản bị bãi bỏ:")
for item in abolished:
    print(f"  📌 {item['so_hieu']} (xuất hiện {item['count']} lần)")
    print(f"     Snippet: {item['snippet'][:100]}...")

---
## 6. Xoá Dữ liệu khỏi ChromaDB

In [ ]:
# ============================================================
# 6.1 Hàm xoá theo số hiệu văn bản (xoá tất cả chunks của văn bản đó)
# ============================================================

def delete_by_doc_no(so_hieu: str, dry_run: bool = True) -> Dict:
    """
    Xoá tất cả chunks thuộc văn bản có số hiệu `so_hieu`.
    
    Args:
        so_hieu  : số hiệu văn bản, VD "30/2020/NĐ-CP"
        dry_run  : True = chỉ xem trước, không xoá thực sự
    Returns:
        {so_hieu, found_ids, deleted_count, dry_run}
    """
    if normalize_doc_no(so_hieu) in {normalize_doc_no(x) for x in NEVER_REMOVE}:
        print(f"🛡️  {so_hieu} nằm trong NEVER_REMOVE — KHÔNG xoá.")
        return {"so_hieu": so_hieu, "found_ids": [], "deleted_count": 0, "dry_run": dry_run}

    col = get_collection(COLLECTION_NAMES["legal"])

    # ChromaDB where filter theo metadata
    results = col.get(
        where={"source_doc_no": {"$eq": so_hieu}},
        include=["metadatas"],
    )
    found_ids = results["ids"]

    print(f"🔍 Tìm kiếm: '{so_hieu}'")
    print(f"   Tìm thấy: {len(found_ids)} chunks")

    if not found_ids:
        print("   Không có gì để xoá.")
        return {"so_hieu": so_hieu, "found_ids": [], "deleted_count": 0, "dry_run": dry_run}

    # Hiển thị sample metadata
    print("   Sample (3 chunks đầu):")
    for meta in results["metadatas"][:3]:
        print(f"     - {meta.get('article', '?')[:60]}")

    if dry_run:
        print(f"   [DRY RUN] Sẽ xoá {len(found_ids)} chunks — đặt dry_run=False để thực hiện.")
        return {"so_hieu": so_hieu, "found_ids": found_ids, "deleted_count": 0, "dry_run": True}

    # Thực sự xoá — xoá theo batch 100 để tránh timeout
    deleted = 0
    BATCH = 100
    for i in range(0, len(found_ids), BATCH):
        col.delete(ids=found_ids[i:i+BATCH])
        deleted += len(found_ids[i:i+BATCH])

    print(f"   ✅ Đã xoá {deleted} chunks của '{so_hieu}'")
    return {"so_hieu": so_hieu, "found_ids": found_ids, "deleted_count": deleted, "dry_run": False}


# ============================================================
# 6.2 Hàm xoá theo Điều cụ thể của một văn bản
# ============================================================

def delete_by_article(so_hieu: str, article_query: str, dry_run: bool = True) -> Dict:
    """
    Xoá chunks thuộc Điều cụ thể của một văn bản.
    
    Args:
        so_hieu       : số hiệu văn bản, VD "30/2020/NĐ-CP"
        article_query : tên/số Điều, VD "Điều 5" hoặc "Điều 5. Quyền"
        dry_run       : True = chỉ xem trước
    """
    col = get_collection(COLLECTION_NAMES["legal"])

    # Lấy tất cả chunks của văn bản
    results = col.get(
        where={"source_doc_no": {"$eq": so_hieu}},
        include=["metadatas"],
    )

    if not results["ids"]:
        print(f"❌ Không tìm thấy văn bản '{so_hieu}' trong ChromaDB.")
        return {"found_ids": [], "deleted_count": 0}

    # Lọc theo article (so khớp mềm)
    matched_ids = []
    article_norm = article_query.lower().strip()
    for cid, meta in zip(results["ids"], results["metadatas"]):
        art = meta.get("article", "").lower()
        if article_norm in art:
            matched_ids.append((cid, meta.get("article", "")))

    print(f"🔍 Tìm kiếm Điều: '{article_query}' trong '{so_hieu}'")
    print(f"   Tìm thấy: {len(matched_ids)} chunks")

    if not matched_ids:
        print("   Không khớp — thử lại với tên Điều khác.")
        print("   Các Điều có trong DB:")
        seen_articles = set()
        for meta in results["metadatas"][:20]:
            art = meta.get("article", "")
            if art not in seen_articles:
                print(f"     · {art[:70]}")
                seen_articles.add(art)
        return {"found_ids": [], "deleted_count": 0}

    print("   Các chunks sẽ bị xoá:")
    for cid, art in matched_ids:
        print(f"     - [{cid[:12]}...] {art[:60]}")

    if dry_run:
        print(f"   [DRY RUN] Sẽ xoá {len(matched_ids)} chunks — đặt dry_run=False để thực hiện.")
        return {"found_ids": [x[0] for x in matched_ids], "deleted_count": 0, "dry_run": True}

    ids_to_delete = [x[0] for x in matched_ids]
    col.delete(ids=ids_to_delete)
    print(f"   ✅ Đã xoá {len(ids_to_delete)} chunks")
    return {"found_ids": ids_to_delete, "deleted_count": len(ids_to_delete), "dry_run": False}


print("✅ Delete functions defined.")

## 7. Upsert Dữ liệu mới vào ChromaDB

In [ ]:
# ============================================================
# 7.1 Hàm upsert chunks vào ChromaDB
# ============================================================

def upsert_chunks(chunks: List[Dict], upsert_batch: int = 64) -> Dict:
    """
    Embed và upsert list chunks vào collection legal_chunks.
    Dùng upsert (không phải add) để idempotent: gọi nhiều lần = kết quả giống nhau.
    
    Args:
        chunks       : list dict từ chunk_article()
        upsert_batch : số chunks mỗi lần upsert
    Returns:
        {upserted, skipped, errors}
    """
    col = get_collection(COLLECTION_NAMES["legal"])
    upserted = 0
    errors   = []

    LEGAL_META_COLS = [
        "doc_id", "source_doc_no", "ministry", "type_normalized",
        "doc_name", "chapter_id", "chapter_name", "article",
        "chunk_index", "total_chunks", "split_type", "word_count",
    ]

    for i in range(0, len(chunks), upsert_batch):
        batch = chunks[i:i+upsert_batch]
        texts = [c["text"] for c in batch]

        try:
            embeddings = embed_passages(texts)
        except Exception as e:
            errors.append(f"Embed batch {i}: {e}")
            continue

        ids = [
            make_chroma_id(c["doc_id"], c["chunk_index"], c["text"])
            for c in batch
        ]
        metadatas = [
            coerce_metadata({k: c.get(k) for k in LEGAL_META_COLS})
            for c in batch
        ]

        try:
            col.upsert(ids=ids, embeddings=embeddings.tolist(),
                       documents=texts, metadatas=metadatas)
            upserted += len(batch)
            print(f"   Upserted batch {i//upsert_batch + 1}: {len(batch)} chunks")
        except Exception as e:
            errors.append(f"Upsert batch {i}: {e}")

    return {"upserted": upserted, "errors": errors}


print("✅ Upsert function defined.")

---
## 8. Pipeline Tổng thể — Nhận văn bản OCR, xử lý & cập nhật DB

In [ ]:
# ============================================================
# 8.1 Full pipeline: OCR text → Validate → Chunking → Detect bãi bỏ → DB Update
# ============================================================

def process_new_document(
    ocr_text: str,
    ministry: str = "",
    dry_run_delete: bool = True,   # True = chỉ preview xoá, không thực thi
    skip_upsert: bool   = False,   # True = không thêm văn bản mới
    manual_so_hieu: str = "",      # Override nếu OCR không nhận diện được
    manual_loai_vb: str = "",      # Override loại văn bản
) -> Dict:
    """
    Pipeline đầy đủ xử lý văn bản OCR mới:
      1. Trích header (số hiệu, loại, tên)
      2. Tách Điều
      3. Chunking
      4. Detect văn bản bị bãi bỏ
      5. Xoá văn bản bãi bỏ khỏi ChromaDB
      6. Upsert văn bản mới vào ChromaDB
    
    Trả về report dict.
    """
    report = {
        "header"          : {},
        "articles_found"  : 0,
        "chunks_created"  : 0,
        "abolished_found" : [],
        "delete_results"  : [],
        "upsert_result"   : {},
        "errors"          : [],
    }

    print("=" * 60)
    print("🚀 Bắt đầu xử lý văn bản OCR")
    print("=" * 60)

    # ── BƯỚC 1: Trích header ───────────────────────────────────────────────
    header = extract_doc_header(ocr_text)
    if manual_so_hieu:
        header["so_hieu"] = manual_so_hieu
    if manual_loai_vb:
        header["loai_vb"] = manual_loai_vb

    report["header"] = header
    print(f"\n📋 Header:")
    print(f"   Số hiệu  : {header['so_hieu'] or '(chưa nhận diện — cần manual_so_hieu)'}")
    print(f"   Loại VB  : {header['loai_vb']}")
    print(f"   Tên VB   : {header['ten_van_ban'][:80] or '(chưa nhận diện)'}")

    if not header["so_hieu"]:
        report["errors"].append("Không nhận diện được số hiệu văn bản — cần truyền manual_so_hieu")
        print("\n⚠️  Không tìm thấy số hiệu — pipeline dừng lại.")
        print("    Dùng: process_new_document(text, manual_so_hieu='XX/YYYY/NĐ-CP')")
        return report

    # ── BƯỚC 2: Tách Điều ─────────────────────────────────────────────────
    print(f"\n📖 Tách Điều...")
    articles = split_articles(ocr_text)
    report["articles_found"] = len(articles)
    print(f"   → {len(articles)} Điều")

    if not articles:
        report["errors"].append("Không tìm thấy Điều nào trong văn bản")
        return report

    # ── BƯỚC 3: Chunking ──────────────────────────────────────────────────
    print(f"\n✂️  Chunking...")
    all_chunks = []
    for art in articles:
        chunks = chunk_article(
            art,
            so_hieu   = header["so_hieu"],
            loai_vb   = header["loai_vb"],
            ten_vb    = header["ten_van_ban"],
            ministry  = ministry,
        )
        all_chunks.extend(chunks)

    report["chunks_created"] = len(all_chunks)
    wc = [c["word_count"] for c in all_chunks]
    print(f"   → {len(all_chunks)} chunks | min={min(wc)} max={max(wc)} mean={sum(wc)//len(wc)} từ")

    # ── BƯỚC 4: Detect bãi bỏ ────────────────────────────────────────────
    print(f"\n🔎 Kiểm tra văn bản bị bãi bỏ...")
    abolished = detect_abolished_docs(ocr_text)
    report["abolished_found"] = abolished

    if not abolished:
        print("   Không phát hiện văn bản bị bãi bỏ.")
    else:
        print(f"   ⚠️  Phát hiện {len(abolished)} văn bản bị bãi bỏ:")
        for item in abolished:
            print(f"   📌 {item['so_hieu']} (đề cập {item['count']} lần)")
            print(f"      Snippet: {item['snippet'][:100]}")

    # ── BƯỚC 5: Xoá văn bản bãi bỏ ───────────────────────────────────────
    if abolished:
        print(f"\n🗑️  Xoá văn bản bãi bỏ khỏi ChromaDB (dry_run={dry_run_delete})...")
        for item in abolished:
            del_result = delete_by_doc_no(item["so_hieu"], dry_run=dry_run_delete)
            report["delete_results"].append(del_result)

    # ── BƯỚC 6: Upsert văn bản mới ────────────────────────────────────────
    if skip_upsert:
        print(f"\n⏭️  skip_upsert=True — bỏ qua bước thêm văn bản mới.")
    else:
        print(f"\n📤 Upsert {len(all_chunks)} chunks vào ChromaDB...")
        upsert_result = upsert_chunks(all_chunks)
        report["upsert_result"] = upsert_result
        if upsert_result["errors"]:
            print(f"   ⚠️  Lỗi: {upsert_result['errors']}")
        else:
            print(f"   ✅ Đã upsert {upsert_result['upserted']} chunks")

    print("\n" + "=" * 60)
    print("✅ Xử lý hoàn tất")
    print("=" * 60)
    return report


print("✅ process_new_document() defined.")

---
## 9. Test Pipeline — Chạy với văn bản OCR mẫu

In [ ]:
# ============================================================
# 9.1 Kiểm tra trạng thái ChromaDB trước khi thực hiện
# ============================================================

def db_status() -> None:
    print("📊 Trạng thái ChromaDB hiện tại:")
    for key, col_name in COLLECTION_NAMES.items():
        try:
            col = get_collection(col_name)
            print(f"   {col_name}: {col.count():,} chunks")
        except Exception as e:
            print(f"   {col_name}: ❌ {e}")

db_status()

In [ ]:
# ============================================================
# 9.2 Test: Chạy pipeline với văn bản OCR mẫu
#
# HƯỚNG DẪN:
#   - Thay SAMPLE_OCR bằng text từ file OCR thực tế của bạn
#   - Lần đầu: để dry_run_delete=True để xem trước
#   - Sau khi xác nhận: đổi dry_run_delete=False
# ============================================================

SAMPLE_OCR_NEW = """
CHÍNH PHỦ
________
Số: 01/2025/NĐ-CP

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
________________________
Hà Nội, ngày 01 tháng 01 năm 2025

NGHỊ ĐỊNH
VỀ QUY ĐỊNH CHI TIẾT THI HÀNH MỘT SỐ ĐIỀU CỦA LUẬT BAN HÀNH VĂN BẢN QUY PHẠM PHÁP LUẬT

Chương I. QUY ĐỊNH CHUNG

Điều 1. Phạm vi điều chỉnh
Nghị định này quy định chi tiết khoản 2 Điều 14, khoản 5 Điều 19 của Luật Ban hành
văn bản quy phạm pháp luật số 80/2015/QH13 đã được sửa đổi, bổ sung một số điều
theo Luật số 63/2020/QH14.

Điều 2. Đối tượng áp dụng
Nghị định này áp dụng đối với:
a) Cơ quan, tổ chức nhà nước;
b) Cán bộ, công chức, viên chức;
c) Tổ chức, cá nhân có liên quan.

Điều 3. Nguyên tắc xây dựng, ban hành văn bản
1. Tuân thủ quy định của Hiến pháp và pháp luật.
2. Bảo đảm tính hợp hiến, hợp pháp, thống nhất, đồng bộ của hệ thống pháp luật.
3. Bảo đảm tính minh bạch, công khai trong quá trình xây dựng văn bản.

Chương II. TRÌNH TỰ, THỦ TỤC XÂY DỰNG VĂN BẢN

Điều 10. Lập đề nghị xây dựng văn bản
1. Cơ quan đề xuất lập hồ sơ đề nghị gồm:
a) Tờ trình đề nghị xây dựng văn bản;
b) Báo cáo đánh giá tác động chính sách;
c) Dự thảo văn bản.
2. Hồ sơ được gửi đến Bộ Tư pháp để thẩm định.

Điều 20. Hiệu lực thi hành
Nghị định này có hiệu lực thi hành kể từ ngày ký.
Bãi bỏ Nghị định số 34/2016/NĐ-CP ngày 14 tháng 5 năm 2016 quy định chi tiết
một số điều của Luật ban hành văn bản quy phạm pháp luật.
Các bộ trưởng, thủ trưởng cơ quan ngang bộ chịu trách nhiệm thi hành Nghị định này.
"""

# ── Chạy pipeline ──────────────────────────────────────────────────────────
report = process_new_document(
    ocr_text        = SAMPLE_OCR_NEW,
    ministry        = "Chính phủ",
    dry_run_delete  = True,   # ← Đổi thành False để xoá thực sự
    skip_upsert     = True,   # ← Đổi thành False để thêm vào DB
)

In [ ]:
# Xem báo cáo chi tiết
print("\n📄 BÁO CÁO XỬ LÝ:")
print(f"  Số hiệu    : {report['header'].get('so_hieu', 'N/A')}")
print(f"  Số Điều    : {report['articles_found']}")
print(f"  Số chunks  : {report['chunks_created']}")
print(f"  Văn bản bãi bỏ phát hiện: {len(report['abolished_found'])}")
print(f"  Upsert     : {report['upsert_result'].get('upserted', 0)} chunks")
print(f"  Lỗi        : {report.get('errors', []) + report['upsert_result'].get('errors', [])}")

---
## 10. Xoá Thủ công — Nhập số hiệu / Điều cụ thể

In [ ]:
# ============================================================
# 10.1 XOÁ THEO SỐ HIỆU VĂN BẢN
#
# Ví dụ: xoá tất cả chunks của Nghị định 110/2004/NĐ-CP
# ============================================================

# ── BƯỚC 1: Preview (dry_run=True) ────────────────────────────────────────
result_preview = delete_by_doc_no(
    so_hieu = "110/2004/NĐ-CP",  # ← Thay bằng số hiệu cần xoá
    dry_run = True,
)

print(f"\nSố chunks sẽ bị xoá: {len(result_preview['found_ids'])}")

In [ ]:
# ── BƯỚC 2: Xác nhận và xoá thực sự (bỏ comment dòng dưới) ───────────────

# result_delete = delete_by_doc_no(
#     so_hieu = "110/2004/NĐ-CP",   # ← Thay bằng số hiệu cần xoá
#     dry_run = False,               # ← Xoá thực sự
# )
# print(f"Đã xoá: {result_delete['deleted_count']} chunks")

print("ℹ️  Bỏ comment cell trên để xoá thực sự.")

In [ ]:
# ============================================================
# 10.2 XOÁ THEO ĐIỀU CỤ THỂ
#
# Ví dụ: chỉ xoá Điều 5 của một văn bản
# ============================================================

# ── BƯỚC 1: Preview ─────────────────────────────────────────────────────────
result_art_preview = delete_by_article(
    so_hieu       = "30/2020/NĐ-CP",   # ← Thay bằng số hiệu văn bản
    article_query = "Điều 5",           # ← Thay bằng tên/số Điều cần xoá
    dry_run       = True,
)

In [ ]:
# ── BƯỚC 2: Xác nhận và xoá thực sự (bỏ comment dòng dưới) ───────────────

# result_art_delete = delete_by_article(
#     so_hieu       = "30/2020/NĐ-CP",   # ← Thay bằng số hiệu văn bản
#     article_query = "Điều 5",           # ← Thay bằng tên/số Điều cần xoá
#     dry_run       = False,              # ← Xoá thực sự
# )

print("ℹ️  Bỏ comment cell trên để xoá thực sự.")

---
## 11. Xoá hàng loạt từ danh sách số hiệu

In [ ]:
# ============================================================
# 11.1 Xoá nhiều văn bản cùng lúc từ danh sách
# ============================================================

def batch_delete_docs(so_hieu_list: List[str], dry_run: bool = True) -> pd.DataFrame:
    """
    Xoá nhiều văn bản từ danh sách số hiệu.
    Trả về DataFrame tổng kết.
    """
    rows = []
    print(f"{'[DRY RUN] ' if dry_run else ''}Xử lý {len(so_hieu_list)} văn bản...\n")

    for so_hieu in so_hieu_list:
        res = delete_by_doc_no(so_hieu, dry_run=dry_run)
        rows.append({
            "so_hieu"      : so_hieu,
            "found_chunks" : len(res["found_ids"]),
            "deleted"      : res["deleted_count"],
            "protected"    : normalize_doc_no(so_hieu) in {normalize_doc_no(x) for x in NEVER_REMOVE},
        })

    df = pd.DataFrame(rows)
    print("\n📊 Kết quả:")
    print(df.to_string(index=False))
    print(f"\n   Tổng chunks {'sẽ bị' if dry_run else 'đã'} xoá: {df['deleted' if not dry_run else 'found_chunks'].sum()}")
    return df


# ── Ví dụ sử dụng: preview danh sách văn bản cần xoá ────────────────────────
DOCS_TO_DELETE = [
    "110/2004/NĐ-CP",
    "09/2010/QĐ-TTg",
    "30/2020/NĐ-CP",   # ← Sẽ bị chặn vì nằm trong NEVER_REMOVE
]

summary_df = batch_delete_docs(DOCS_TO_DELETE, dry_run=True)

In [ ]:
# ── Xoá thực sự (bỏ comment) ──────────────────────────────────────────────

# summary_df = batch_delete_docs(DOCS_TO_DELETE, dry_run=False)

print("ℹ️  Bỏ comment cell trên để xoá thực sự.")

---
## 12. Verify — Kiểm tra sau mỗi thao tác

In [ ]:
# ============================================================
# 12.1 Kiểm tra số chunks của một văn bản cụ thể
# ============================================================

def verify_doc(so_hieu: str, show_articles: bool = True) -> None:
    """Xem thông tin một văn bản trong ChromaDB."""
    col = get_collection(COLLECTION_NAMES["legal"])

    results = col.get(
        where={"source_doc_no": {"$eq": so_hieu}},
        include=["metadatas"],
    )

    print(f"📋 Văn bản: {so_hieu}")
    print(f"   Tổng chunks: {len(results['ids'])}")

    if not results["ids"]:
        print("   (Không có trong DB)")
        return

    if show_articles:
        # Group theo Điều
        articles_seen: Dict[str, int] = {}
        for meta in results["metadatas"]:
            art = meta.get("article", "(không rõ)")
            articles_seen[art] = articles_seen.get(art, 0) + 1

        print(f"   Số Điều: {len(articles_seen)}")
        print("   Danh sách Điều:")
        for art, cnt in sorted(articles_seen.items(),
                               key=lambda x: x[0])[:20]:  # hiện max 20
            print(f"     {cnt:2d} chunk(s)  {art[:70]}")
        if len(articles_seen) > 20:
            print(f"     ... và {len(articles_seen)-20} Điều khác")


# ── Kiểm tra văn bản quan trọng ──────────────────────────────────────────────
verify_doc("30/2020/NĐ-CP")

In [ ]:
# Kiểm tra trạng thái tổng thể sau khi thực hiện các thay đổi
db_status()

---
## 13. (Tuỳ chọn) Rebuild BM25 Index sau khi cập nhật DB

> ⚠️ Chạy cell này **sau khi đã upsert/xoá xong** để đồng bộ BM25 index với ChromaDB.

In [ ]:
# ============================================================
# 13.1 Hướng dẫn rebuild BM25
# Notebook này không tự rebuild BM25 vì:
#   - BM25 cần toàn bộ legal_chunks.parquet (~130MB)
#   - Rebuild mất ~5-10 phút cho 240k chunks
#   - hybridRetrieval.ipynb có sẵn hàm build_bm25_index(..., force_rebuild=True)
# ============================================================

print("""📋 Hướng dẫn rebuild BM25 sau khi cập nhật DB:

  OPTION 1 — Rebuild trong hybridRetrieval.ipynb:
    Đặt FORCE_REBUILD = True trong cell "5. Build & Load BM25 Indexes"
    → Tự động rebuild và cache vào dataset/bm25/bm25_legal_v2.pkl

  OPTION 2 — Xoá cache, rebuild tự động khi chạy lại:
    import os
    os.remove('dataset/bm25/bm25_legal_v2.pkl')
    os.remove('dataset/bm25/bm25_legal_v2.meta.pkl')
    → Lần chạy tiếp theo sẽ tự rebuild

  OPTION 3 — Chỉ rebuild nếu có thay đổi lớn (>1% số chunks):
    Với update nhỏ (vài văn bản), BM25 cũ vẫn hoạt động đủ tốt.
    BM25 không có văn bản mới → chỉ mất keyword recall của văn bản đó.
    Dense search (ChromaDB) vẫn tìm được document mới ngay lập tức.
""")

In [ ]:
# ── (Optional) Xoá BM25 cache để force rebuild ─────────────────────────────

import os

BM25_FILES = [
    PROJECT_ROOT / "dataset" / "bm25" / "bm25_legal_v2.pkl",
    PROJECT_ROOT / "dataset" / "bm25" / "bm25_legal_v2.meta.pkl",
]

# Bỏ comment để xoá cache
# for f in BM25_FILES:
#     if f.exists():
#         os.remove(f)
#         print(f"  Đã xoá: {f}")
#     else:
#         print(f"  Không tồn tại: {f}")

print("BM25 cache status:")
for f in BM25_FILES:
    status = f"✅ {f.stat().st_size/1024/1024:.1f} MB" if f.exists() else "❌ (chưa có)"
    print(f"  {status}  {f.name}")